# Step 01_04_04 -- Identity Resolution (aoestats)

**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_04 -- Data Cleaning
**Step:** 01_04_04
**Dataset:** aoestats
**Predecessor:** 01_04_03 (Minimal Cross-Dataset History View -- complete)
**Step scope (exploration only):** aoestats has no nickname column in any raw
table (structural asymmetry confirmed in 01_01_02). profile_id (DOUBLE in
players_raw; BIGINT in player_history_all) is the sole identity signal.
Invariant I2 (lowercased nickname as canonical) is natively unmeetable.
This step quantifies: (A) sentinel/NULL profile_id health, (B) per-profile
activity distribution, (C) duplicate census, (D) rating-trajectory monotonicity
probe, (E) replay_summary_raw parseability, (F) civ-fingerprint JSD as
behavioural surrogate, (G) cross-dataset feasibility preview to aoec.
**No new VIEWs, no raw-table modifications, no schema YAML changes (I9).**
**Invariants applied:**
  - I2 (absence of nickname documents natively unmeetable canonical ID)
  - I6 (all SQL verbatim in artifact sql_queries block)
  - I7 (all thresholds documented with provenance -- see inline comments)
  - I9 (exploration only -- no upstream modification)
**Date:** 2026-04-18

## Cell 1 -- Imports

In [1]:
import ast
import bisect
import json
from collections import defaultdict
from datetime import date
from pathlib import Path

import numpy as np
from scipy.spatial.distance import jensenshannon

from rts_predict.common.notebook_utils import (
    get_notebook_db,
    get_reports_dir,
    setup_notebook_logging,
)

logger = setup_notebook_logging()
print("Imports complete.")

Imports complete.


## Cell 2 -- DuckDB Connection (read-only)

This step is exploration only. A read-only connection is used.

In [2]:
from rts_predict.games.aoe2.config import AOE2COMPANION_DB_FILE

db = get_notebook_db("aoe2", "aoestats", read_only=True)
con = db.con
print(f"DuckDB connection opened (read-only).")
print(f"aoec DB path: {AOE2COMPANION_DB_FILE}")

DuckDB connection opened (read-only).
aoec DB path: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/data/db/db.duckdb


## Cell 3 -- Reports directory + artifact paths

In [3]:
reports_dir = get_reports_dir("aoe2", "aoestats")
artifacts_dir = reports_dir / "artifacts" / "01_exploration" / "04_cleaning"
artifacts_dir.mkdir(parents=True, exist_ok=True)

JSON_OUT = artifacts_dir / "01_04_04_identity_resolution.json"
MD_OUT = artifacts_dir / "01_04_04_identity_resolution.md"

print(f"JSON artifact: {JSON_OUT}")
print(f"MD artifact:   {MD_OUT}")

JSON artifact: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/reports/artifacts/01_exploration/04_cleaning/01_04_04_identity_resolution.json
MD artifact:   /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/reports/artifacts/01_exploration/04_cleaning/01_04_04_identity_resolution.md


## Cell 4 -- Task A: Sentinel + NULL audit (4 columns x 3 tables/views)

I7 provenance for sentinel values:
  - profile_id = 0: 01_02_04 univariate census zero_count = 0 in players_raw
    (confirmed below -- 0 zero values). No -1 sentinel documented.
  - The -1 sentinel for PROFILE_IDs is an aoec-specific finding (12.95M AI rows
    per 01_03_02); aoestats has no such sentinel per census evidence.

In [4]:
TASK_A_SQL = {}

# --- players_raw.profile_id (DOUBLE) ---
sql_a1 = """
SELECT
  COUNT(*)                                        AS n_rows,
  COUNT(*) FILTER (WHERE profile_id IS NULL)      AS n_null,
  COUNT(*) FILTER (WHERE profile_id = 0)          AS n_zero,
  COUNT(*) FILTER (WHERE profile_id < 0)          AS n_negative,
  COUNT(*) FILTER (WHERE profile_id = -1)         AS n_minus_one_sentinel,
  MIN(profile_id)                                 AS min_val,
  MAX(profile_id)                                 AS max_val,
  COUNT(DISTINCT profile_id)                      AS cardinality
FROM players_raw
"""
TASK_A_SQL["players_raw_profile_id"] = sql_a1
r_a1 = con.execute(sql_a1).fetchone()
print("players_raw.profile_id (DOUBLE):")
print(f"  n_rows={r_a1[0]}, n_null={r_a1[1]}, n_zero={r_a1[2]}, "
      f"n_negative={r_a1[3]}, n_minus_one={r_a1[4]}, "
      f"min={r_a1[5]}, max={r_a1[6]}, cardinality={r_a1[7]}")

# --- player_history_all.profile_id (BIGINT) ---
sql_a2 = """
SELECT
  COUNT(*)                                        AS n_rows,
  COUNT(*) FILTER (WHERE profile_id IS NULL)      AS n_null,
  COUNT(*) FILTER (WHERE profile_id = 0)          AS n_zero,
  COUNT(*) FILTER (WHERE profile_id < 0)          AS n_negative,
  COUNT(*) FILTER (WHERE profile_id = -1)         AS n_minus_one_sentinel,
  MIN(profile_id)                                 AS min_val,
  MAX(profile_id)                                 AS max_val,
  COUNT(DISTINCT profile_id)                      AS cardinality
FROM player_history_all
"""
TASK_A_SQL["player_history_all_profile_id"] = sql_a2
r_a2 = con.execute(sql_a2).fetchone()
print("player_history_all.profile_id (BIGINT):")
print(f"  n_rows={r_a2[0]}, n_null={r_a2[1]}, n_zero={r_a2[2]}, "
      f"n_negative={r_a2[3]}, n_minus_one={r_a2[4]}, "
      f"min={r_a2[5]}, max={r_a2[6]}, cardinality={r_a2[7]}")

# --- matches_1v1_clean.p0_profile_id (BIGINT) ---
sql_a3 = """
SELECT
  COUNT(*)                                          AS n_rows,
  COUNT(*) FILTER (WHERE p0_profile_id IS NULL)     AS n_null,
  COUNT(*) FILTER (WHERE p0_profile_id = 0)         AS n_zero,
  COUNT(*) FILTER (WHERE p0_profile_id < 0)         AS n_negative,
  COUNT(*) FILTER (WHERE p0_profile_id = -1)        AS n_minus_one_sentinel,
  MIN(p0_profile_id)                                AS min_val,
  MAX(p0_profile_id)                                AS max_val,
  COUNT(DISTINCT p0_profile_id)                     AS cardinality
FROM matches_1v1_clean
"""
TASK_A_SQL["matches_1v1_clean_p0_profile_id"] = sql_a3
r_a3 = con.execute(sql_a3).fetchone()
print("matches_1v1_clean.p0_profile_id (BIGINT):")
print(f"  n_rows={r_a3[0]}, n_null={r_a3[1]}, n_zero={r_a3[2]}, "
      f"n_negative={r_a3[3]}, n_minus_one={r_a3[4]}, "
      f"min={r_a3[5]}, max={r_a3[6]}, cardinality={r_a3[7]}")

# --- matches_1v1_clean.p1_profile_id (BIGINT) ---
sql_a4 = """
SELECT
  COUNT(*)                                          AS n_rows,
  COUNT(*) FILTER (WHERE p1_profile_id IS NULL)     AS n_null,
  COUNT(*) FILTER (WHERE p1_profile_id = 0)         AS n_zero,
  COUNT(*) FILTER (WHERE p1_profile_id < 0)         AS n_negative,
  COUNT(*) FILTER (WHERE p1_profile_id = -1)        AS n_minus_one_sentinel,
  MIN(p1_profile_id)                                AS min_val,
  MAX(p1_profile_id)                                AS max_val,
  COUNT(DISTINCT p1_profile_id)                     AS cardinality
FROM matches_1v1_clean
"""
TASK_A_SQL["matches_1v1_clean_p1_profile_id"] = sql_a4
r_a4 = con.execute(sql_a4).fetchone()
print("matches_1v1_clean.p1_profile_id (BIGINT):")
print(f"  n_rows={r_a4[0]}, n_null={r_a4[1]}, n_zero={r_a4[2]}, "
      f"n_negative={r_a4[3]}, n_minus_one={r_a4[4]}, "
      f"min={r_a4[5]}, max={r_a4[6]}, cardinality={r_a4[7]}")

# Compile Task A results
task_a_results = {
    "players_raw.profile_id": {
        "type": "DOUBLE", "n_rows": r_a1[0], "n_null": r_a1[1],
        "n_zero": r_a1[2], "n_negative": r_a1[3], "n_minus_one_sentinel": r_a1[4],
        "min": float(r_a1[5]) if r_a1[5] is not None else None,
        "max": float(r_a1[6]) if r_a1[6] is not None else None,
        "cardinality": r_a1[7],
    },
    "player_history_all.profile_id": {
        "type": "BIGINT", "n_rows": r_a2[0], "n_null": r_a2[1],
        "n_zero": r_a2[2], "n_negative": r_a2[3], "n_minus_one_sentinel": r_a2[4],
        "min": r_a2[5], "max": r_a2[6], "cardinality": r_a2[7],
    },
    "matches_1v1_clean.p0_profile_id": {
        "type": "BIGINT", "n_rows": r_a3[0], "n_null": r_a3[1],
        "n_zero": r_a3[2], "n_negative": r_a3[3], "n_minus_one_sentinel": r_a3[4],
        "min": r_a3[5], "max": r_a3[6], "cardinality": r_a3[7],
    },
    "matches_1v1_clean.p1_profile_id": {
        "type": "BIGINT", "n_rows": r_a4[0], "n_null": r_a4[1],
        "n_zero": r_a4[2], "n_negative": r_a4[3], "n_minus_one_sentinel": r_a4[4],
        "min": r_a4[5], "max": r_a4[6], "cardinality": r_a4[7],
    },
}
print("\nTask A: PASS -- No NULL, zero, negative, or -1 sentinel values in "
      "player_history_all / matches_1v1_clean. players_raw has 1,185 NULLs "
      "(excluded by player_history_all filter).")

players_raw.profile_id (DOUBLE):
  n_rows=107627584, n_null=1185, n_zero=0, n_negative=0, n_minus_one=0, min=18.0, max=24853897.0, cardinality=641662


player_history_all.profile_id (BIGINT):
  n_rows=107626399, n_null=0, n_zero=0, n_negative=0, n_minus_one=0, min=18, max=24853897, cardinality=641662


matches_1v1_clean.p0_profile_id (BIGINT):
  n_rows=17814947, n_null=0, n_zero=0, n_negative=0, n_minus_one=0, min=18, max=24853897, cardinality=310670


matches_1v1_clean.p1_profile_id (BIGINT):
  n_rows=17814947, n_null=0, n_zero=0, n_negative=0, n_minus_one=0, min=18, max=24853301, cardinality=309727

Task A: PASS -- No NULL, zero, negative, or -1 sentinel values in player_history_all / matches_1v1_clean. players_raw has 1,185 NULLs (excluded by player_history_all filter).


## Cell 5 -- Task B: Per-profile activity distribution (player_history_all)

Quantile thresholds (>=20 matches, >=180 active days) are defined in Task F
based on the observed distribution. No magic numbers here -- all reported
counts are descriptive.

In [5]:
TASK_B_SQL = {}

sql_b1 = """
WITH profile_stats AS (
  SELECT
    profile_id,
    COUNT(*)                                                AS match_count,
    COUNT(DISTINCT CAST(started_timestamp AS DATE))         AS active_days
  FROM player_history_all
  GROUP BY profile_id
)
SELECT
  COUNT(*)                                        AS total_profiles,
  APPROX_QUANTILE(match_count, 0.25)             AS match_q25,
  APPROX_QUANTILE(match_count, 0.50)             AS match_q50,
  APPROX_QUANTILE(match_count, 0.75)             AS match_q75,
  APPROX_QUANTILE(match_count, 0.90)             AS match_q90,
  APPROX_QUANTILE(match_count, 0.99)             AS match_q99,
  MAX(match_count)                                AS match_max,
  APPROX_QUANTILE(active_days, 0.25)             AS days_q25,
  APPROX_QUANTILE(active_days, 0.50)             AS days_q50,
  APPROX_QUANTILE(active_days, 0.75)             AS days_q75,
  APPROX_QUANTILE(active_days, 0.90)             AS days_q90,
  APPROX_QUANTILE(active_days, 0.99)             AS days_q99,
  MAX(active_days)                                AS days_max,
  COUNT(*) FILTER (WHERE active_days = 1)        AS single_day_profiles
FROM profile_stats
"""
TASK_B_SQL["profile_activity_distribution"] = sql_b1
r_b1 = con.execute(sql_b1).fetchone()
print("Per-profile activity distribution:")
print(f"  total_profiles: {r_b1[0]:,}")
print(f"  match_count quantiles: q25={r_b1[1]} q50={r_b1[2]} "
      f"q75={r_b1[3]} q90={r_b1[4]} q99={r_b1[5]} max={r_b1[6]}")
print(f"  active_days quantiles: q25={r_b1[7]} q50={r_b1[8]} "
      f"q75={r_b1[9]} q90={r_b1[10]} q99={r_b1[11]} max={r_b1[12]}")
print(f"  single_day_profiles: {r_b1[13]:,}")

sql_b2 = """
WITH profile_ladders AS (
  SELECT
    p.profile_id,
    COUNT(DISTINCT m.leaderboard)   AS n_leaderboards
  FROM players_raw p
  JOIN matches_raw m ON p.game_id = m.game_id
  WHERE p.profile_id IS NOT NULL
  GROUP BY p.profile_id
)
SELECT
  COUNT(*) FILTER (WHERE n_leaderboards = 1)  AS single_ladder_profiles,
  COUNT(*) FILTER (WHERE n_leaderboards > 1)  AS multi_ladder_profiles,
  MAX(n_leaderboards)                          AS max_leaderboards
FROM profile_ladders
"""
TASK_B_SQL["multi_ladder_profiles"] = sql_b2
r_b2 = con.execute(sql_b2).fetchone()
print(f"\n  single_ladder_profiles: {r_b2[0]:,}")
print(f"  multi_ladder_profiles: {r_b2[1]:,}")
print(f"  max_leaderboards: {r_b2[2]}")

task_b_results = {
    "total_profiles": r_b1[0],
    "match_count_quantiles": {
        "q25": r_b1[1], "q50": r_b1[2], "q75": r_b1[3],
        "q90": r_b1[4], "q99": r_b1[5], "max": r_b1[6],
    },
    "active_days_quantiles": {
        "q25": r_b1[7], "q50": r_b1[8], "q75": r_b1[9],
        "q90": r_b1[10], "q99": r_b1[11], "max": r_b1[12],
    },
    "single_day_profiles": r_b1[13],
    "single_ladder_profiles": r_b2[0],
    "multi_ladder_profiles": r_b2[1],
    "max_leaderboards": r_b2[2],
}

Per-profile activity distribution:
  total_profiles: 641,662
  match_count quantiles: q25=2 q50=13 q75=116 q90=479 q99=2152 max=15075
  active_days quantiles: q25=1 q50=6 q75=41 q90=148 q99=519 max=1064
  single_day_profiles: 160,163



  single_ladder_profiles: 372,555
  multi_ladder_profiles: 269,107
  max_leaderboards: 4


## Cell 6 -- Task C: Duplicate (game_id, profile_id) census in players_raw

Uses census-aligned COALESCE string-concatenation key methodology,
identical to 01_03_01 systematic_profile.json `duplicate_players_rows` field.
Anchor: 489 from 01_03_01 (artifact: reports/artifacts/01_exploration/
03_profiling/01_03_01_systematic_profile.json).
HALT condition: result outside 479-499 (489 +/- 10 tolerance).

In [6]:
TASK_C_SQL = {}

sql_c = """
WITH keyed AS (
  SELECT
    CAST(game_id AS VARCHAR)
      || '_'
      || COALESCE(CAST(profile_id AS VARCHAR), '__NULL__')  AS row_key
  FROM players_raw
)
SELECT COUNT(*) - COUNT(DISTINCT row_key)  AS duplicate_rows
FROM keyed
"""
TASK_C_SQL["duplicate_game_id_profile_id"] = sql_c
r_c = con.execute(sql_c).fetchone()
dup_count = r_c[0]
print(f"Duplicate (game_id, profile_id) rows (census-aligned COALESCE key): {dup_count}")

ANCHOR_489 = 489
TOLERANCE = 10
in_range = abs(dup_count - ANCHOR_489) <= TOLERANCE
status_c = "PASS" if in_range else "HALT"
print(f"Anchor check (489 +/- {TOLERANCE}): {status_c} "
      f"(drift = {dup_count - ANCHOR_489:+d})")

assert in_range, (
    f"HALT: Duplicate count {dup_count} outside anchor range "
    f"{ANCHOR_489 - TOLERANCE}..{ANCHOR_489 + TOLERANCE}"
)

task_c_results = {
    "duplicate_rows": dup_count,
    "anchor": ANCHOR_489,
    "tolerance": TOLERANCE,
    "drift": dup_count - ANCHOR_489,
    "status": status_c,
}

Duplicate (game_id, profile_id) rows (census-aligned COALESCE key): 489
Anchor check (489 +/- 10): PASS (drift = +0)


## Cell 7 -- Task D: Rating-trajectory monotonicity probe

10,000-profile reservoir sample, seed=20260418 (deterministic).
I7 provenance for |delta| > 500 threshold:
  Anecdotal RTS convention for a "suspiciously large" single-game rating swing.
  A typical AoE2 rating system awards ±16-32 per game (Elo K-factor); 500 is
  ~15x the expected swing. This is a first-cut sanity bound, not a calibrated
  threshold. Treat as exploratory, not dispositive.

In [7]:
TASK_D_SQL = {}

sql_d = """
WITH profile_sample AS (
  SELECT DISTINCT profile_id
  FROM player_history_all
  USING SAMPLE RESERVOIR(10000 ROWS) REPEATABLE (20260418)
),
trajectory AS (
  SELECT
    p.profile_id,
    p.old_rating,
    LAG(p.old_rating) OVER (
      PARTITION BY p.profile_id
      ORDER BY p.started_timestamp
    ) AS prev_rating
  FROM player_history_all p
  INNER JOIN profile_sample ps ON p.profile_id = ps.profile_id
),
deltas AS (
  SELECT ABS(old_rating - prev_rating)  AS abs_delta
  FROM trajectory
  WHERE prev_rating IS NOT NULL
)
SELECT
  COUNT(*)                                              AS n_deltas,
  COUNT(*) FILTER (WHERE abs_delta > 500)               AS n_large_delta,
  APPROX_QUANTILE(abs_delta, 0.50)                     AS median_abs_delta,
  APPROX_QUANTILE(abs_delta, 0.75)                     AS p75_abs_delta,
  APPROX_QUANTILE(abs_delta, 0.99)                     AS p99_abs_delta,
  MAX(abs_delta)                                        AS max_abs_delta
FROM deltas
"""
TASK_D_SQL["rating_trajectory_monotonicity"] = sql_d
r_d = con.execute(sql_d).fetchone()
print("Rating-trajectory monotonicity probe (10,000-profile reservoir, seed=20260418):")
print(f"  n_deltas: {r_d[0]:,}")
print(f"  n_large_delta (|delta| > 500): {r_d[1]:,}")
print(f"  median_abs_delta: {r_d[2]}")
print(f"  p75_abs_delta: {r_d[3]}")
print(f"  p99_abs_delta: {r_d[4]}")
print(f"  max_abs_delta: {r_d[5]}")
print(f"  large_delta_rate: {r_d[1] / r_d[0] * 100:.3f}%")
print("  NOTE: |delta| > 500 threshold is anecdotal RTS convention "
      "(~15x expected K-factor swing); hedge as first-cut sanity bound.")

task_d_results = {
    "seed": 20260418,
    "reservoir_size": 10000,
    "n_deltas": r_d[0],
    "n_large_delta_gt_500": r_d[1],
    "large_delta_rate_pct": round(r_d[1] / r_d[0] * 100, 4),
    "median_abs_delta": r_d[2],
    "p75_abs_delta": r_d[3],
    "p99_abs_delta": r_d[4],
    "max_abs_delta": r_d[5],
    "threshold_500_provenance": (
        "Anecdotal RTS convention: ~15x expected Elo K-factor swing. "
        "First-cut sanity bound, not calibrated."
    ),
}

Rating-trajectory monotonicity probe (10,000-profile reservoir, seed=20260418):
  n_deltas: 11,311,282
  n_large_delta (|delta| > 500): 12,047
  median_abs_delta: 16
  p75_abs_delta: 17
  p99_abs_delta: 226
  max_abs_delta: 1444
  large_delta_rate: 0.107%
  NOTE: |delta| > 500 threshold is anecdotal RTS convention (~15x expected K-factor swing); hedge as first-cut sanity bound.


## Cell 8 -- Task E: replay_summary_raw JSON-parseability probe

Feasibility probe on 1,000-row sample (seed=20260418).
FEASIBILITY ONLY -- do not extract player names.
Note: 01_03_03 (t04_overviews_and_replay) established non_empty_pct=13.95%.
The field stores Python dict format (single quotes), NOT valid JSON.
Parseable with ast.literal_eval. Contains 'age_stats' and 'opening_name' keys.

In [8]:
TASK_E_SQL = {}

sql_e = """
SELECT replay_summary_raw, LENGTH(replay_summary_raw) AS len_chars
FROM players_raw
WHERE replay_summary_raw IS NOT NULL
  AND replay_summary_raw != '{}'
  AND LENGTH(replay_summary_raw) > 2
USING SAMPLE RESERVOIR(1000 ROWS) REPEATABLE (20260418)
"""
TASK_E_SQL["replay_summary_raw_sample"] = sql_e
e_rows = con.execute(sql_e).fetchall()
print(f"Non-empty replay_summary_raw rows in 1000-row sample: {len(e_rows)}")

parseable = 0
unparseable = 0
key_counts: dict = {}
lengths = [r[1] for r in e_rows]

for val, ln in e_rows:
    try:
        parsed = ast.literal_eval(val)
        parseable += 1
        if isinstance(parsed, dict):
            for k in parsed.keys():
                key_counts[k] = key_counts.get(k, 0) + 1
    except Exception:
        unparseable += 1

mean_len = sum(lengths) / len(lengths) if lengths else 0.0
max_len = max(lengths) if lengths else 0

print(f"  Parseable (ast.literal_eval): {parseable}")
print(f"  Unparseable: {unparseable}")
print(f"  Mean length: {mean_len:.1f} chars")
print(f"  Max length: {max_len} chars")
print(f"  Top keys: {sorted(key_counts.items(), key=lambda x: -x[1])[:5]}")
print("  FORMAT: Python dict (single-quote keys), NOT valid JSON.")
print("  FEASIBILITY: Contains 'opening_name' key -- name extraction feasible "
      "via ast.literal_eval but deferred (out of scope).")

task_e_results = {
    "sample_size_non_empty": len(e_rows),
    "parseable_ast": parseable,
    "unparseable": unparseable,
    "format": "Python dict (single-quote keys; ast.literal_eval required)",
    "mean_length_chars": round(mean_len, 1),
    "max_length_chars": max_len,
    "top_keys": sorted(key_counts.items(), key=lambda x: -x[1]),
    "feasibility_note": (
        "opening_name key present in all parseable rows. Name extraction is "
        "technically feasible via ast.literal_eval. DEFERRED: this step does "
        "not extract names (scope: feasibility probe only)."
    ),
}

Non-empty replay_summary_raw rows in 1000-row sample: 146
  Parseable (ast.literal_eval): 146
  Unparseable: 0
  Mean length: 1288.5 chars
  Max length: 2236 chars
  Top keys: [('age_stats', 146), ('opening_name', 146)]
  FORMAT: Python dict (single-quote keys), NOT valid JSON.
  FEASIBILITY: Contains 'opening_name' key -- name extraction feasible via ast.literal_eval but deferred (out of scope).


## Cell 9 -- Task F: Civ-fingerprint JSD analysis

Within-profile: first-half vs second-half 50-dim civ-frequency JSD.
Cross-profile: 10,000 random pairs as control.
Qualifying threshold: >= 20 matches AND >= 180 active days.
  - 20 matches: minimum to estimate a 50-dim distribution; < 20 gives too few
    observations per civ bin for a stable first-half/second-half split.
  - 180 active days: ensures the player's activity spans at least one season
    (AoE2 meta patches every ~3-6 months); captures potential civ-preference
    drift that the JSD would detect.
  - Both thresholds are exploratory heuristics for this analysis, not cleaning
    decisions. They are documented here per I7 and flagged for calibration.

I7 hedge on JSD thresholds (0.10/0.30/0.50):
  These thresholds derive from symmetric KL divergence interpretation.
  JSD = 0.10 ~ subtle shift; 0.30 ~ moderate; 0.50 ~ substantial.
  Hahn et al. 2020 (SC2 APM/build-order) is ADJACENT literature, not direct.
  Civ-marginal JSD over <= 43 civs (at time of study; 50 now with DLC) with
  meta drift is a COARSE PROXY. Rename/multi-account resolution for aoestats
  remains unsolved pending cross-dataset bridge.

In [9]:
TASK_F_SQL = {}

sql_f_civs = """
SELECT DISTINCT civ FROM players_raw WHERE civ IS NOT NULL ORDER BY civ
"""
TASK_F_SQL["distinct_civs"] = sql_f_civs
civs = [r[0] for r in con.execute(sql_f_civs).fetchall()]
civ_idx = {c: i for i, c in enumerate(civs)}
n_civs = len(civs)
print(f"Total distinct civs: {n_civs}")

sql_f_qualify = """
WITH profile_stats AS (
  SELECT
    profile_id,
    COUNT(*)                                        AS match_count,
    COUNT(DISTINCT CAST(started_timestamp AS DATE)) AS active_days
  FROM player_history_all
  GROUP BY profile_id
)
SELECT profile_id, match_count, active_days
FROM profile_stats
WHERE match_count >= 20 AND active_days >= 180
ORDER BY profile_id
"""
TASK_F_SQL["qualifying_profiles"] = sql_f_qualify
qualifying = con.execute(sql_f_qualify).fetchall()
print(f"Qualifying profiles (>= 20 matches AND >= 180 active days): {len(qualifying):,}")

Total distinct civs: 50


Qualifying profiles (>= 20 matches AND >= 180 active days): 52,455


## Cell 10 -- Task F continued: Within-profile JSD computation

In [10]:
q_profiles = [r[0] for r in qualifying]
within_jsd = []
BATCH_SIZE = 500

for batch_start in range(0, len(q_profiles), BATCH_SIZE):
    batch = q_profiles[batch_start : batch_start + BATCH_SIZE]
    batch_str = ", ".join(str(p) for p in batch)

    sql_batch = f"""
    SELECT profile_id, civ, CAST(started_timestamp AS VARCHAR) AS ts_str
    FROM player_history_all
    WHERE profile_id IN ({batch_str}) AND civ IS NOT NULL
    ORDER BY profile_id, started_timestamp
    """
    rows = con.execute(sql_batch).fetchall()

    profile_matches: dict = defaultdict(list)
    for pid, civ, ts in rows:
        profile_matches[pid].append(civ)

    for pid, match_civs in profile_matches.items():
        n = len(match_civs)
        if n < 4:
            continue
        mid = n // 2
        p = np.zeros(n_civs)
        q_vec = np.zeros(n_civs)
        for c in match_civs[:mid]:
            if c in civ_idx:
                p[civ_idx[c]] += 1
        for c in match_civs[mid:]:
            if c in civ_idx:
                q_vec[civ_idx[c]] += 1
        if p.sum() > 0:
            p = p / p.sum()
        if q_vec.sum() > 0:
            q_vec = q_vec / q_vec.sum()
        within_jsd.append(float(jensenshannon(p, q_vec) ** 2))

within_arr = np.array(within_jsd)
print(f"Within-profile JSD computed for {len(within_jsd):,} profiles.")

within_quantiles = {}
for q_pct, label in [(0.05, "p5"), (0.25, "p25"), (0.50, "p50"),
                     (0.75, "p75"), (0.95, "p95"), (0.99, "p99")]:
    val = float(np.quantile(within_arr, q_pct))
    within_quantiles[label] = round(val, 4)
    print(f"  {label}: {val:.4f}")

within_below_010 = int((within_arr < 0.10).sum())
within_010_030 = int(((within_arr >= 0.10) & (within_arr < 0.30)).sum())
within_030_050 = int(((within_arr >= 0.30) & (within_arr < 0.50)).sum())
within_above_050 = int((within_arr >= 0.50).sum())
print(f"\n  Threshold breakdown:")
print(f"    < 0.10: {within_below_010:,} ({within_below_010/len(within_jsd):.1%})")
print(f"    0.10-0.30: {within_010_030:,} ({within_010_030/len(within_jsd):.1%})")
print(f"    0.30-0.50: {within_030_050:,} ({within_030_050/len(within_jsd):.1%})")
print(f"    >= 0.50: {within_above_050:,} ({within_above_050/len(within_jsd):.1%})")

Within-profile JSD computed for 52,455 profiles.
  p5: 0.0270
  p25: 0.0725
  p50: 0.1262
  p75: 0.1993
  p95: 0.3472
  p99: 0.4800

  Threshold breakdown:
    < 0.10: 19,976 (38.1%)
    0.10-0.30: 27,984 (53.3%)
    0.30-0.50: 4,113 (7.8%)
    >= 0.50: 382 (0.7%)


## Cell 11 -- Task F continued: Cross-profile JSD control (10,000 random pairs)

In [11]:
rng = np.random.default_rng(20260418)
sample_size = min(300, len(q_profiles))
sampled_idx = rng.choice(len(q_profiles), size=sample_size, replace=False)
sampled_pids = [q_profiles[i] for i in sampled_idx]

pid_str = ", ".join(str(p) for p in sampled_pids)
sql_cross = f"""
SELECT profile_id, civ
FROM player_history_all
WHERE profile_id IN ({pid_str}) AND civ IS NOT NULL
ORDER BY profile_id
"""
TASK_F_SQL["cross_profile_sample_data"] = sql_cross
cross_rows = con.execute(sql_cross).fetchall()

profile_civ_dist: dict = {}
profile_matches_cross: dict = defaultdict(list)
for pid, civ in cross_rows:
    profile_matches_cross[pid].append(civ)

for pid, match_civs in profile_matches_cross.items():
    p = np.zeros(n_civs)
    for c in match_civs:
        if c in civ_idx:
            p[civ_idx[c]] += 1
    if p.sum() > 0:
        p = p / p.sum()
    profile_civ_dist[pid] = p

pid_list = list(profile_civ_dist.keys())
cross_jsd = []
pairs_done = 0
while pairs_done < 10000:
    i, j = rng.choice(len(pid_list), size=2, replace=False)
    if i == j:
        continue
    pa = profile_civ_dist[pid_list[i]]
    pb = profile_civ_dist[pid_list[j]]
    cross_jsd.append(float(jensenshannon(pa, pb) ** 2))
    pairs_done += 1

cross_arr = np.array(cross_jsd)
print(f"Cross-profile JSD computed for {len(cross_jsd):,} random pairs.")

cross_quantiles = {}
for q_pct, label in [(0.05, "p5"), (0.25, "p25"), (0.50, "p50"),
                     (0.75, "p75"), (0.95, "p95"), (0.99, "p99")]:
    val = float(np.quantile(cross_arr, q_pct))
    cross_quantiles[label] = round(val, 4)
    print(f"  {label}: {val:.4f}")

Cross-profile JSD computed for 10,000 random pairs.
  p5: 0.0998
  p25: 0.2288
  p50: 0.3606
  p75: 0.4885
  p95: 0.6257
  p99: 0.6711


## Cell 12 -- Task F continued: 10-profile concrete example table

In [12]:
top10_pids = [
    (3288518, 15075, 741), (4216671, 11230, 977), (14527552, 10949, 834),
    (15369793, 10205, 693), (5600740, 10199, 881), (11962723, 10151, 729),
    (9958595, 9398, 871), (6297927, 9167, 919), (15802329, 8861, 543),
    (5864544, 8752, 1009),
]

example_table = []
for pid, mc, ad in top10_pids:
    sql_ex = f"""
    SELECT civ, CAST(started_timestamp AS VARCHAR) AS ts_str
    FROM player_history_all
    WHERE profile_id = {pid} AND civ IS NOT NULL
    ORDER BY started_timestamp
    """
    ex_rows = con.execute(sql_ex).fetchall()
    match_civs = [r[0] for r in ex_rows]
    n = len(match_civs)
    mid = n // 2
    p = np.zeros(n_civs)
    q_vec = np.zeros(n_civs)
    for c in match_civs[:mid]:
        if c in civ_idx:
            p[civ_idx[c]] += 1
    for c in match_civs[mid:]:
        if c in civ_idx:
            q_vec[civ_idx[c]] += 1
    if p.sum() > 0:
        p = p / p.sum()
    if q_vec.sum() > 0:
        q_vec = q_vec / q_vec.sum()
    jsd_val = float(jensenshannon(p, q_vec) ** 2)
    example_table.append({
        "profile_id": pid, "match_count": mc, "active_days": ad,
        "jsd": round(jsd_val, 4),
        "top_civ_first_half": civs[int(np.argmax(p))] if p.sum() > 0 else "N/A",
        "pct_first_half": round(float(p.max()), 3),
        "top_civ_second_half": civs[int(np.argmax(q_vec))] if q_vec.sum() > 0 else "N/A",
        "pct_second_half": round(float(q_vec.max()), 3),
    })

print("10-profile example table (top by match_count):")
for row in example_table:
    print(f"  pid={row['profile_id']} mc={row['match_count']} "
          f"ad={row['active_days']} JSD={row['jsd']:.4f} "
          f"top1st={row['top_civ_first_half']}({row['pct_first_half']:.1%}) "
          f"top2nd={row['top_civ_second_half']}({row['pct_second_half']:.1%})")

task_f_results = {
    "n_civs": n_civs,
    "qualifying_threshold": {"min_matches": 20, "min_active_days": 180},
    "n_qualifying_profiles": len(qualifying),
    "within_profile_jsd": {
        "n_profiles": len(within_jsd),
        "quantiles": within_quantiles,
        "below_010": within_below_010,
        "range_010_030": within_010_030,
        "range_030_050": within_030_050,
        "above_050": within_above_050,
    },
    "cross_profile_jsd": {
        "n_pairs": len(cross_jsd),
        "quantiles": cross_quantiles,
        "sample_profiles_used": sample_size,
    },
    "example_table": example_table,
    "i7_hedge": (
        "Thresholds 0.10/0.30/0.50 derive from symmetric KL divergence interpretation. "
        "Hahn et al. 2020 (SC2 APM/build-order) is adjacent literature, not direct. "
        "Civ-marginal JSD over <= 50 civs with meta drift is a coarse proxy. "
        "Rename/multi-account resolution for aoestats remains unsolved pending "
        "cross-dataset bridge."
    ),
}

10-profile example table (top by match_count):
  pid=3288518 mc=15075 ad=741 JSD=0.2419 top1st=malay(36.7%) top2nd=malay(32.3%)
  pid=4216671 mc=11230 ad=977 JSD=0.1320 top1st=franks(19.1%) top2nd=vikings(18.9%)
  pid=14527552 mc=10949 ad=834 JSD=0.1334 top1st=saracens(8.1%) top2nd=dravidians(19.5%)
  pid=15369793 mc=10205 ad=693 JSD=0.1131 top1st=mongols(47.7%) top2nd=spanish(27.6%)
  pid=5600740 mc=10199 ad=881 JSD=0.1856 top1st=goths(20.9%) top2nd=celts(10.1%)
  pid=11962723 mc=10151 ad=729 JSD=0.1448 top1st=khmer(14.7%) top2nd=ethiopians(14.6%)
  pid=9958595 mc=9398 ad=871 JSD=0.0425 top1st=spanish(48.6%) top2nd=spanish(34.1%)
  pid=6297927 mc=9167 ad=919 JSD=0.0789 top1st=franks(16.2%) top2nd=franks(10.2%)
  pid=15802329 mc=8861 ad=543 JSD=0.1137 top1st=gurjaras(20.4%) top2nd=gurjaras(27.1%)
  pid=5864544 mc=8752 ad=1009 JSD=0.1352 top1st=persians(16.2%) top2nd=koreans(15.9%)


## Cell 13 -- Task G: Cross-dataset feasibility preview

Window: 2026-01-25..2026-01-31 (intersection of aoestats and aoec coverage).
aoestats rm_1v1: leaderboard='random_map' (confirmed matches_raw DISTINCT).
aoec rm_1v1: internalLeaderboardId IN (6, 18) (RM and RM unranked equivalents).
Blocker: 60s temporal + civ-set equality + 50-ELO proximity.

I7 provenance for blocking thresholds:
  - 60s temporal window: conventional replay-submission delay between aoestats
    and aoec APIs; exploratory heuristic.
  - 50-ELO proximity: derived from aoestats 01_03_03 finding that avg_elo is
    within 0.5 ELO of player average (max_abs_deviation=0.0 for 1v1 scope);
    50 is ~100x conservative relative to that finding.
  - MATERIALIZED CTE required: DuckDB RESERVOIR(N) on a TIMESTAMPTZ-filtered
    query samples from the full table before the WHERE clause is applied;
    materializing the filtered set first ensures correct 1000-row sample.

Verdict rubric (CI-aware, from plan):
  A = strong  if CI lower bound > 0.85 AND >= 30 filtered matches
  B = partial if CI overlaps [0.10, 0.85] OR 10 <= filtered < 30
  C = disjoint if CI upper bound < 0.10 OR < 10 filtered

In [13]:
TASK_G_SQL = {}

EPOCH_START = 1769295600   # 2026-01-25 00:00:00 UTC
EPOCH_END   = 1769900400   # 2026-02-01 00:00:00 UTC
WINDOW_SECS = 60
ELO_BAND    = 50
SEED        = 20260418

con.execute(f"ATTACH '{AOE2COMPANION_DB_FILE}' AS aoec (READ_ONLY)")
print(f"Attached aoec DB as READ_ONLY.")

sql_g_aoestats = f"""
WITH window_matches AS MATERIALIZED (
  SELECT
    game_id,
    EPOCH(started_timestamp AT TIME ZONE 'UTC')    AS epoch_secs,
    p0_profile_id,
    p1_profile_id,
    LOWER(COALESCE(p0_civ, ''))                    AS p0_civ,
    LOWER(COALESCE(p1_civ, ''))                    AS p1_civ,
    p0_old_rating,
    p1_old_rating
  FROM matches_1v1_clean
  WHERE EPOCH(started_timestamp AT TIME ZONE 'UTC') >= {EPOCH_START}
    AND EPOCH(started_timestamp AT TIME ZONE 'UTC') < {EPOCH_END}
)
SELECT * FROM window_matches
USING SAMPLE RESERVOIR(1000 ROWS) REPEATABLE ({SEED})
"""
TASK_G_SQL["aoestats_rm_1v1_sample"] = sql_g_aoestats
aoestats_sample = con.execute(sql_g_aoestats).fetchall()
print(f"aoestats rm_1v1 sample in window: {len(aoestats_sample)} matches")

sql_g_aoec = f"""
SELECT
  matchId,
  EPOCH(started)                              AS epoch_secs,
  profileId,
  LOWER(COALESCE(civ, ''))                    AS civ,
  rating
FROM aoec.matches_raw
WHERE internalLeaderboardId IN (6, 18)
  AND status = 'player'
  AND EPOCH(started) >= {EPOCH_START}
  AND EPOCH(started) < {EPOCH_END}
  AND profileId IS NOT NULL
  AND profileId != -1
"""
TASK_G_SQL["aoec_rm_1v1_window"] = sql_g_aoec
aoec_rows = con.execute(sql_g_aoec).fetchall()
print(f"aoec rm_1v1 player rows in window: {len(aoec_rows):,}")

Attached aoec DB as READ_ONLY.


aoestats rm_1v1 sample in window: 1000 matches


aoec rm_1v1 player rows in window: 386,018


## Cell 14 -- Task G continued: Blocking + agreement computation

In [14]:
aoec_by_match: dict = defaultdict(list)
for matchId, epoch, profileId, civ, rating in aoec_rows:
    aoec_by_match[matchId].append(
        {"epoch": float(epoch), "profileId": int(profileId),
         "civ": civ, "rating": rating}
    )

aoec_1v1 = {mid: pl for mid, pl in aoec_by_match.items() if len(pl) == 2}
print(f"aoec 1v1 matches (2 players exactly): {len(aoec_1v1):,}")

aoec_list = []
for mid, players in aoec_1v1.items():
    epoch = players[0]["epoch"]
    civ_set = frozenset(p["civ"] for p in players)
    pids = frozenset(p["profileId"] for p in players)
    ratings = sorted(p["rating"] for p in players if p["rating"] is not None)
    aoec_list.append({"mid": mid, "epoch": epoch,
                       "civ_set": civ_set, "pids": pids, "ratings": ratings})

aoec_list.sort(key=lambda x: x["epoch"])
aoec_epochs = [a["epoch"] for a in aoec_list]

block_hits = 0
filtered_hits = 0
profile_id_agreements = []

for row in aoestats_sample:
    game_id, epoch, p0_pid, p1_pid, p0_civ, p1_civ, p0_rating, p1_rating = row
    epoch = float(epoch)
    aoestats_civ_set = frozenset([p0_civ, p1_civ])
    aoestats_pids = frozenset([int(p0_pid), int(p1_pid)])
    aoestats_ratings = sorted(
        r for r in [p0_rating, p1_rating] if r is not None
    )

    lo = bisect.bisect_left(aoec_epochs, epoch - WINDOW_SECS)
    hi = bisect.bisect_right(aoec_epochs, epoch + WINDOW_SECS)

    for candidate in aoec_list[lo:hi]:
        block_hits += 1
        if candidate["civ_set"] != aoestats_civ_set:
            continue
        cand_r = candidate["ratings"]
        if len(cand_r) != 2 or len(aoestats_ratings) != 2:
            continue
        if not (abs(aoestats_ratings[0] - cand_r[0]) <= ELO_BAND
                and abs(aoestats_ratings[1] - cand_r[1]) <= ELO_BAND):
            continue
        filtered_hits += 1
        profile_id_agreements.append(
            1 if aoestats_pids == candidate["pids"] else 0
        )

n_sample = len(aoestats_sample)
print(f"Blocking results:")
print(f"  n_aoestats_sample: {n_sample}")
print(f"  block_hits (<=60s temporal): {block_hits:,}")
print(f"  block_hit_rate: {block_hits / n_sample:.4f}")
print(f"  filtered_matches (civ+ELO): {filtered_hits}")
print(f"  filtered_match_rate: {filtered_hits / n_sample:.4f}")
print(f"  n_with_profile_comparison: {len(profile_id_agreements)}")

aoec 1v1 matches (2 players exactly): 193,009


Blocking results:
  n_aoestats_sample: 1000
  block_hits (<=60s temporal): 41,998
  block_hit_rate: 41.9980
  filtered_matches (civ+ELO): 993
  filtered_match_rate: 0.9930
  n_with_profile_comparison: 993


## Cell 15 -- Task G continued: Bootstrap CI + verdict

In [15]:
n_filt = len(profile_id_agreements)
if n_filt > 0:
    agreement_rate = sum(profile_id_agreements) / n_filt
    print(f"  profile_id_agreement_rate: {agreement_rate:.4f}")
    print(f"  n_agreements: {sum(profile_id_agreements)}")

    rng_g = np.random.default_rng(SEED)
    arr_g = np.array(profile_id_agreements)
    boot_means = [
        rng_g.choice(arr_g, size=n_filt, replace=True).mean()
        for _ in range(10000)
    ]
    ci_lo = float(np.percentile(boot_means, 2.5))
    ci_hi = float(np.percentile(boot_means, 97.5))
    print(f"  95% bootstrap CI: [{ci_lo:.4f}, {ci_hi:.4f}]")

    # Rubric: A=strong if CI lower > 0.85 AND >=30 filtered
    #         B=partial if CI overlaps [0.10, 0.85] OR 10<=filtered<30
    #         C=disjoint if CI upper < 0.10 OR <10 filtered
    if n_filt < 10:
        verdict = "C"
        verdict_reason = f"filtered_count={n_filt} < 10 -- insufficient evidence"
    elif ci_lo > 0.85 and n_filt >= 30:
        verdict = "A"
        verdict_reason = (
            f"CI lower bound {ci_lo:.4f} > 0.85 AND filtered >= 30 "
            "(strong namespace overlap)"
        )
    elif ci_hi < 0.10:
        verdict = "C"
        verdict_reason = f"CI upper bound {ci_hi:.4f} < 0.10 -- disjoint namespaces"
    else:
        verdict = "B"
        verdict_reason = (
            f"CI [{ci_lo:.4f}, {ci_hi:.4f}] overlaps [0.10, 0.85]"
        )
    print(f"  Verdict: {verdict} -- {verdict_reason}")
else:
    agreement_rate = None
    ci_lo = ci_hi = None
    verdict = "C"
    verdict_reason = "No filtered matches -- disjoint / insufficient evidence"
    print(f"  Verdict: {verdict} -- {verdict_reason}")

# Pre-compute display strings (used in decisions ledger and MD artifact)
agree_rate_str = f"{agreement_rate:.4f}" if agreement_rate is not None else "N/A"
ci_lo_str = f"{ci_lo:.4f}" if ci_lo is not None else "N/A"
ci_hi_str = f"{ci_hi:.4f}" if ci_hi is not None else "N/A"
n_agree_val = sum(profile_id_agreements) if profile_id_agreements else 0
print(f"Display strings: agreement={agree_rate_str} CI=[{ci_lo_str},{ci_hi_str}]")

task_g_results = {
    "window": "2026-01-25T00:00:00Z .. 2026-02-01T00:00:00Z",
    "epoch_start": EPOCH_START,
    "epoch_end": EPOCH_END,
    "aoestats_leaderboard_filter": "leaderboard='random_map' (matches_raw)",
    "aoec_leaderboard_filter": "internalLeaderboardId IN (6, 18)",
    "seed": SEED,
    "n_aoestats_sample": n_sample,
    "n_aoec_1v1_matches": len(aoec_1v1),
    "blocking_secs": WINDOW_SECS,
    "elo_band": ELO_BAND,
    "block_hits": block_hits,
    "block_hit_rate": round(block_hits / n_sample, 4) if n_sample > 0 else 0,
    "filtered_hits": filtered_hits,
    "filtered_match_rate": round(filtered_hits / n_sample, 4) if n_sample > 0 else 0,
    "n_with_profile_comparison": n_filt,
    "profile_id_agreement_rate": (
        round(agreement_rate, 4) if agreement_rate is not None else None
    ),
    "n_agreements": sum(profile_id_agreements) if profile_id_agreements else 0,
    "bootstrap_ci_95": [round(ci_lo, 4), round(ci_hi, 4)]
        if ci_lo is not None else None,
    "verdict": verdict,
    "verdict_reason": verdict_reason,
    "i7_provenance": {
        "60s_temporal_window": (
            "Conventional API submission delay (exploratory heuristic)."
        ),
        "50_elo_band": (
            "Derived from aoestats 01_03_03 finding: avg_elo within 0.5 ELO "
            "of player old_rating average (max_abs_deviation=0.0 for 1v1 scope). "
            "50 is ~100x conservative."
        ),
        "materialized_cte": (
            "DuckDB RESERVOIR(N) on TIMESTAMPTZ WHERE clause samples from full "
            "table; MATERIALIZED CTE forces filter evaluation before sampling."
        ),
    },
}

  profile_id_agreement_rate: 0.9960
  n_agreements: 989
  95% bootstrap CI: [0.9919, 0.9990]
  Verdict: A -- CI lower bound 0.9919 > 0.85 AND filtered >= 30 (strong namespace overlap)
Display strings: agreement=0.9960 CI=[0.9919,0.9990]


## Cell 16 -- DS-AOESTATS-IDENTITY decision ledger

5 decisions covering {identity-key, NULL/sentinel, rename-detection-substitute,
collision, cross-dataset-bridge}.

In [16]:
decisions = [
    {
        "id": "DS-AOESTATS-IDENTITY-01",
        "category": "identity-key",
        "column": "profile_id (players_raw DOUBLE; player_history_all BIGINT; "
                  "matches_1v1_clean p0/p1_profile_id BIGINT)",
        "finding": (
            "profile_id is the sole identity signal. No nickname column exists "
            "in any raw table (14 cols in players_raw, 0 name/display/nick fields). "
            "I2 canonical-nickname invariant is natively unmeetable for aoestats. "
            "profile_id is free of NULL (in player_history_all), zero, negative, "
            "and -1 sentinel values. Cardinality: 641,662 distinct profiles. "
            "Type asymmetry: DOUBLE in players_raw (DuckDB promotion artifact); "
            "BIGINT in all views (safe for integer comparison after CAST)."
        ),
        "recommendation": (
            "USE profile_id (BIGINT) as the Phase 02 entity key for aoestats. "
            "Document absence of nickname. Do not attempt to merge profiles via "
            "handle similarity -- no handle data exists."
        ),
        "action": "Phase 02 design uses profile_id as identity key for aoestats.",
    },
    {
        "id": "DS-AOESTATS-IDENTITY-02",
        "category": "NULL/sentinel",
        "column": "players_raw.profile_id (1,185 NULLs)",
        "finding": (
            "players_raw has 1,185 NULL profile_id rows (0.001% of 107.6M rows). "
            "These are excluded by the player_history_all filter "
            "(profile_id IS NOT NULL). No zero, negative, or -1 sentinel detected "
            "in any profile_id column across all 3 objects audited. "
            "Task C confirms 489 duplicate (game_id, profile_id) rows via "
            "census-aligned COALESCE key -- matching 01_03_01 anchor (drift=0)."
        ),
        "recommendation": (
            "No additional sentinel handling required. The 01_04_00 filter "
            "(profile_id IS NOT NULL) already resolves the NULL population. "
            "489 duplicates are negligible (< 0.001%) and carry no signal for "
            "identity resolution."
        ),
        "action": "No VIEW DDL changes needed. Document in Phase 02 as known clean.",
    },
    {
        "id": "DS-AOESTATS-IDENTITY-03",
        "category": "rename-detection-substitute",
        "column": "civ distribution (player_history_all); "
                  "replay_summary_raw (players_raw)",
        "finding": (
            "Without nickname: civ-fingerprint JSD serves as the best available "
            "behavioural surrogate. Within-profile JSD (first-half vs second-half): "
            f"median={within_quantiles['p50']}, p95={within_quantiles['p95']}. "
            f"Cross-profile JSD control (random pairs): "
            f"median={cross_quantiles['p50']}, p95={cross_quantiles['p95']}. "
            "The within-profile distribution is consistently lower than cross-profile, "
            "confirming temporal stability. However: "
            "Hahn et al. 2020 (SC2 APM/build-order) is adjacent, not direct. "
            "Civ-marginal JSD over <= 50 civs with meta drift is a coarse proxy. "
            "replay_summary_raw is 13.95% non-empty; content is Python dict format "
            "(not JSON); contains 'opening_name' key (name extraction feasible via "
            "ast.literal_eval but deferred -- out of scope for this step)."
        ),
        "recommendation": (
            "Rename/multi-account resolution for aoestats remains unsolved. "
            "Civ-fingerprint JSD is NOT sufficient as a standalone rename detector. "
            "Flag for CROSS PR if cross-dataset bridge (Task G verdict) is pursued. "
            "replay_summary_raw name extraction deferred to dedicated step."
        ),
        "action": (
            "Phase 02 uses profile_id as-is. Record I2 limitation in thesis §4.2.2."
        ),
    },
    {
        "id": "DS-AOESTATS-IDENTITY-04",
        "category": "collision",
        "column": "profile_id (cross-table type collision: DOUBLE vs BIGINT)",
        "finding": (
            "players_raw stores profile_id as DOUBLE (DuckDB Arrow promotion from "
            "Parquet mixed-type weekly files). All analytical views cast to BIGINT. "
            "Range: min=18, max=24,853,897 -- well within BIGINT safe range (no "
            "precision loss in DOUBLE-to-BIGINT cast for integers <= 2^53). "
            "Multi-ladder profiles: 269,107 of 641,662 profiles appear in > 1 "
            "leaderboard (random_map, team_random_map, co_random_map, co_team_rm). "
            "Profile-level collision risk: negligible (global namespace, no "
            "region-scoped ID like sc2egset toon_id)."
        ),
        "recommendation": (
            "Always CAST(profile_id AS BIGINT) when joining players_raw to views. "
            "Multi-ladder profiles are expected -- the same player may have "
            "different rating trajectories per leaderboard. Phase 02 should "
            "scope rating computation to leaderboard='random_map' for 1v1 RM."
        ),
        "action": "Document type cast in Phase 02 feature pipeline docstring.",
    },
    {
        "id": "DS-AOESTATS-IDENTITY-05",
        "category": "cross-dataset-bridge",
        "column": "profile_id (aoestats) vs profileId (aoec)",
        "finding": (
            f"Cross-dataset feasibility (Task G): 1,000-match reservoir from "
            f"2026-01-25..2026-01-31. Blocking: 60s temporal + civ-set + 50-ELO. "
            f"filtered_hits={filtered_hits}, "
            f"profile_id_agreement_rate={agree_rate_str}, "
            f"95% bootstrap CI=[{ci_lo_str}, {ci_hi_str}]. "
            f"Verdict: {verdict} -- {verdict_reason}. "
            "aoestats profile_id and aoe2companion profileId appear to share the "
            "same integer namespace (both from aoe2insights.com API lineage). "
            "This enables a cross-dataset name bridge: aoe2companion.name can "
            "serve as I2 canonical nickname for aoestats via profile_id=profileId join."
        ),
        "recommendation": (
            "Proceed with cross-dataset profile_id bridge for the name resolution "
            "path. Full mapping deferred to CROSS PR (not Phase 01 scope). "
            "Conditional on aoec T06 verdict agreement (symmetric rubric)."
        ),
        "action": (
            "Flag for CROSS PR. Thesis §4.2.2 can note namespace-sharing as "
            "empirically supported by Task G agreement rate."
        ),
    },
]

for d in decisions:
    print(f"\n{d['id']} ({d['category']}): {d['finding'][:80]}...")


DS-AOESTATS-IDENTITY-01 (identity-key): profile_id is the sole identity signal. No nickname column exists in any raw tab...

DS-AOESTATS-IDENTITY-02 (NULL/sentinel): players_raw has 1,185 NULL profile_id rows (0.001% of 107.6M rows). These are ex...

DS-AOESTATS-IDENTITY-03 (rename-detection-substitute): Without nickname: civ-fingerprint JSD serves as the best available behavioural s...

DS-AOESTATS-IDENTITY-04 (collision): players_raw stores profile_id as DOUBLE (DuckDB Arrow promotion from Parquet mix...

DS-AOESTATS-IDENTITY-05 (cross-dataset-bridge): Cross-dataset feasibility (Task G): 1,000-match reservoir from 2026-01-25..2026-...


## Cell 17 -- Write JSON artifact (I6: all SQL verbatim)

In [17]:
artifact = {
    "step": "01_04_04",
    "dataset": "aoestats",
    "date": str(date.today()),
    "task_a_sentinel_null_audit": task_a_results,
    "task_b_activity_distribution": task_b_results,
    "task_c_duplicate_census": task_c_results,
    "task_d_rating_trajectory": task_d_results,
    "task_e_replay_summary": task_e_results,
    "task_f_civ_fingerprint_jsd": task_f_results,
    "task_g_cross_dataset_feasibility": task_g_results,
    "decisions": decisions,
    "sql_queries": {
        **TASK_A_SQL,
        **TASK_B_SQL,
        **TASK_C_SQL,
        **TASK_D_SQL,
        **TASK_E_SQL,
        **TASK_F_SQL,
        **TASK_G_SQL,
    },
    "invariants": {
        "I2": "profile_id is sole identity signal; no nickname column exists (14 "
              "cols in players_raw); I2 natively unmeetable for aoestats.",
        "I6": "All SQL queries stored verbatim in sql_queries block above.",
        "I7": (
            "All thresholds documented: 500 ELO delta (anecdotal sanity bound), "
            "50 ELO blocking band (100x conservative vs 01_03_03 0.5-ELO finding), "
            "60s temporal window (conventional API delay), "
            "0.10/0.30/0.50 JSD (symmetric KL interpretation, hedged)."
        ),
        "I9": "No new VIEWs, no raw-table modifications, no schema YAML changes.",
    },
}

with open(JSON_OUT, "w") as f:
    json.dump(artifact, f, indent=2)
print(f"JSON artifact written: {JSON_OUT}")

JSON artifact written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/reports/artifacts/01_exploration/04_cleaning/01_04_04_identity_resolution.json


## Cell 18 -- Write MD artifact

In [18]:
n_agree = n_agree_val  # defined in Cell 15 after verdict computation

md = f"""# Step 01_04_04 -- Identity Resolution (aoestats)

**Date:** {date.today()}
**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_04 -- Data Cleaning
**Dataset:** aoestats
**Step scope:** Exploratory census of aoestats identity structure.
No new VIEWs, no raw-table modifications, no schema YAML changes (I9).

---

## Structural asymmetry

aoestats has **no nickname column** in any raw table. `players_raw` has 14
columns; no `name`, `display`, or `nickname` field exists. Invariant I2
(lowercased in-game nickname as canonical player identifier) is **natively
unmeetable** for this dataset without a cross-dataset bridge.

`profile_id` (DOUBLE in `players_raw`; BIGINT in views) is the sole identity
signal. This step characterises its health, probes behavioural surrogates, and
assesses cross-dataset bridge feasibility.

---

## Task A -- Sentinel + NULL audit

| Column | Table/View | Type | n_rows | n_null | n_zero | n_negative | n_minus_one | min | max | cardinality |
|---|---|---|---|---|---|---|---|---|---|---|
| profile_id | players_raw | DOUBLE | {r_a1[0]:,} | {r_a1[1]:,} | {r_a1[2]} | {r_a1[3]} | {r_a1[4]} | {r_a1[5]:.0f} | {r_a1[6]:.0f} | {r_a1[7]:,} |
| profile_id | player_history_all | BIGINT | {r_a2[0]:,} | {r_a2[1]} | {r_a2[2]} | {r_a2[3]} | {r_a2[4]} | {r_a2[5]} | {r_a2[6]:,} | {r_a2[7]:,} |
| p0_profile_id | matches_1v1_clean | BIGINT | {r_a3[0]:,} | {r_a3[1]} | {r_a3[2]} | {r_a3[3]} | {r_a3[4]} | {r_a3[5]} | {r_a3[6]:,} | {r_a3[7]:,} |
| p1_profile_id | matches_1v1_clean | BIGINT | {r_a4[0]:,} | {r_a4[1]} | {r_a4[2]} | {r_a4[3]} | {r_a4[4]} | {r_a4[5]} | {r_a4[6]:,} | {r_a4[7]:,} |

**Finding:** All views are clean -- no NULL, zero, negative, or -1 sentinel
values. The 1,185 NULLs in `players_raw.profile_id` are already excluded by
the `player_history_all` filter (`profile_id IS NOT NULL`).
DOUBLE type in `players_raw` is a DuckDB Arrow-promotion artifact; integer
values are exact for the observed range (max=24,853,897 << 2^53).

---

## Task B -- Per-profile activity distribution

| Metric | q25 | q50 | q75 | q90 | q99 | max |
|---|---|---|---|---|---|---|
| match_count | {r_b1[1]} | {r_b1[2]} | {r_b1[3]} | {r_b1[4]} | {r_b1[5]} | {r_b1[6]} |
| active_days | {r_b1[7]} | {r_b1[8]} | {r_b1[9]} | {r_b1[10]} | {r_b1[11]} | {r_b1[12]} |

- **Total profiles:** {r_b1[0]:,}
- **Single-day profiles:** {r_b1[13]:,} (casual/one-off players)
- **Single-ladder profiles:** {r_b2[0]:,}
- **Multi-ladder profiles:** {r_b2[1]:,} (appear in > 1 leaderboard)
- **Max leaderboards per profile:** {r_b2[2]}

---

## Task C -- Duplicate (game_id, profile_id) census

Census-aligned COALESCE key: `CAST(game_id AS VARCHAR) || '_' ||
COALESCE(CAST(profile_id AS VARCHAR), '__NULL__')`.

- **Duplicate rows:** {dup_count} (anchor: 489 from 01_03_01; drift={dup_count - ANCHOR_489:+d})
- **Gate:** PASS -- within 489 ± 10 tolerance.

---

## Task D -- Rating-trajectory monotonicity probe

10,000-profile reservoir sample, seed=20260418. LAG(old_rating) deltas.

| Metric | Value |
|---|---|
| n_deltas | {r_d[0]:,} |
| n_large_delta (\\|Δ\\| > 500) | {r_d[1]:,} ({r_d[1] / r_d[0] * 100:.3f}%) |
| median_abs_delta | {r_d[2]} |
| p75_abs_delta | {r_d[3]} |
| p99_abs_delta | {r_d[4]} |
| max_abs_delta | {r_d[5]} |

**I7 hedge:** The 500-ELO threshold is an anecdotal RTS convention (~15x
expected K-factor swing). This is a first-cut sanity bound, not a calibrated
threshold. The {r_d[1]:,} large-delta observations ({r_d[1] / r_d[0] * 100:.3f}%)
may represent season resets, provisional periods, or data ingestion artifacts.

---

## Task E -- replay_summary_raw parseability probe

Sample: 1,000 rows (seed=20260418) from non-empty `replay_summary_raw` entries.

- **Non-empty rows in sample:** {len(e_rows)}
- **Format:** Python dict (single-quote keys) -- **NOT valid JSON**
- **Parseable (ast.literal_eval):** {parseable} / {len(e_rows)}
- **Mean length:** {mean_len:.1f} chars
- **Max length:** {max_len} chars
- **Top keys:** {dict(sorted(key_counts.items(), key=lambda x: -x[1])[:3])}

**Feasibility verdict:** `opening_name` key is present in all parseable rows.
Name extraction via `ast.literal_eval` is technically feasible.
**DEFERRED -- out of scope for this step.** A dedicated step would be needed
to extract and validate player names from replay_summary_raw.

---

## Task F -- Civ-fingerprint JSD analysis

**Qualifying threshold:** >= 20 matches AND >= 180 active days.
**Qualifying profiles:** {len(qualifying):,} of {r_b1[0]:,} total.

### Within-profile JSD (first-half vs second-half, 50-dim civ distribution)

| p5 | p25 | p50 | p75 | p95 | p99 |
|---|---|---|---|---|---|
| {within_quantiles['p5']} | {within_quantiles['p25']} | {within_quantiles['p50']} | {within_quantiles['p75']} | {within_quantiles['p95']} | {within_quantiles['p99']} |

Threshold breakdown (JSD < 0.10 = stable; 0.10-0.30 = moderate drift;
0.30-0.50 = substantial; >= 0.50 = high drift):

| JSD < 0.10 | 0.10 -- 0.30 | 0.30 -- 0.50 | >= 0.50 |
|---|---|---|---|
| {within_below_010:,} ({within_below_010/len(within_jsd):.1%}) | {within_010_030:,} ({within_010_030/len(within_jsd):.1%}) | {within_030_050:,} ({within_030_050/len(within_jsd):.1%}) | {within_above_050:,} ({within_above_050/len(within_jsd):.1%}) |

### Cross-profile JSD (10,000 random pairs, control)

| p5 | p25 | p50 | p75 | p95 | p99 |
|---|---|---|---|---|---|
| {cross_quantiles['p5']} | {cross_quantiles['p25']} | {cross_quantiles['p50']} | {cross_quantiles['p75']} | {cross_quantiles['p95']} | {cross_quantiles['p99']} |

**Separation:** Within-profile median ({within_quantiles['p50']}) <<
cross-profile median ({cross_quantiles['p50']}), confirming temporal stability
of individual civ preferences relative to random pairs.

### 10-profile example table (top by match_count)

| profile_id | match_count | active_days | JSD | top_civ_1st | % | top_civ_2nd | % |
|---|---|---|---|---|---|---|---|
"""

for row in example_table:
    md += (
        f"| {row['profile_id']} | {row['match_count']:,} | {row['active_days']} "
        f"| {row['jsd']:.4f} | {row['top_civ_first_half']} "
        f"| {row['pct_first_half']:.1%} | {row['top_civ_second_half']} "
        f"| {row['pct_second_half']:.1%} |\n"
    )

md += f"""
### Hedge (I7 + R1-WARNING-6 fix)

Hahn et al. 2020 (SC2 APM/build-order) is **adjacent literature, not direct**.
Civ-marginal JSD over <= 50 civs with meta drift is a **coarse proxy** for
identity resolution. Thresholds 0.10/0.30/0.50 are symmetric KL divergence
interpretations; per-corpus calibration is required before operational use.
**Rename/multi-account resolution for aoestats remains unsolved pending the
cross-dataset bridge evaluated in Task G.**

---

## Task G -- Cross-dataset feasibility preview

**Window:** 2026-01-25..2026-01-31 (aoestats × aoec coverage intersection).
**aoestats filter:** `leaderboard='random_map'` (matches_1v1_clean).
**aoec filter:** `internalLeaderboardId IN (6, 18)` (rm_1v1 equivalent).
**Blocker:** {WINDOW_SECS}s temporal + civ-set equality + {ELO_BAND}-ELO proximity.

| Metric | Value |
|---|---|
| n_aoestats_sample | {n_sample:,} |
| n_aoec_1v1_matches_in_window | {len(aoec_1v1):,} |
| block_hits (<=60s) | {block_hits:,} ({block_hits / n_sample:.4f} per sampled match) |
| filtered_matches (civ+ELO) | {filtered_hits:,} |
| filtered_match_rate | {filtered_hits / n_sample:.4f} |
| n_with_profile_comparison | {n_filt} |
| profile_id_agreement_rate | {agree_rate_str} |
| n_agreements | {n_agree} |
| 95% bootstrap CI | [{ci_lo_str}, {ci_hi_str}] |
| **Verdict** | **{verdict}** |

**Verdict rubric (CI-aware):**
- A = strong if CI lower bound > 0.85 AND >= 30 filtered matches
- B = partial if CI overlaps [0.10, 0.85] OR 10 <= filtered < 30
- C = disjoint if CI upper bound < 0.10 OR < 10 filtered

**Verdict: {verdict} -- {verdict_reason}**

**I7 provenance for blocking thresholds:**
- 60s temporal: conventional API submission delay (exploratory heuristic).
- 50-ELO band: derived from 01_03_03 finding (max_abs_deviation=0.0 for 1v1
  scope); 50 is ~100x conservative.

---

## Decision Ledger -- DS-AOESTATS-IDENTITY-01..05

"""

for d in decisions:
    md += f"### {d['id']} ({d['category']})\n\n"
    md += f"**Column:** {d['column']}\n\n"
    md += f"**Finding:** {d['finding']}\n\n"
    md += f"**Recommendation:** {d['recommendation']}\n\n"
    md += f"**Action:** {d['action']}\n\n"
    md += "---\n\n"

md += f"""## Synthesis

1. **Identity key:** `profile_id` (BIGINT) is clean, sentinel-free, and globally
   scoped (no region sharding unlike sc2egset toon_id). Use as Phase 02 entity key.

2. **I2 limitation:** aoestats has no nickname column. I2 is natively unmeetable
   without a cross-dataset bridge. This is documented as a dataset characteristic.

3. **Civ-fingerprint stability:** Within-profile JSD (median={within_quantiles['p50']})
   is substantially lower than cross-profile control (median={cross_quantiles['p50']}),
   confirming temporal civ-preference stability. However, this cannot serve as a
   rename detector without ground truth labels.

4. **replay_summary_raw:** Contains `opening_name` (Python dict format). Name
   extraction is feasible but deferred to a dedicated step.

5. **Cross-dataset bridge (Task G):** Verdict {verdict}. aoestats `profile_id`
   and aoe2companion `profileId` appear to share the same integer namespace
   (agreement rate {agree_rate_str}, 95% CI [{ci_lo_str}, {ci_hi_str}]).
   A direct join on profile_id=profileId would provide aoe2companion player names
   for aoestats profiles. Full cross-dataset mapping is a CROSS PR deliverable.

**Scientific invariants:** I2 (natively unmeetable -- documented). I6 (all SQL
verbatim in JSON). I7 (all thresholds provenance documented). I9 (no upstream
modification).
"""

with open(MD_OUT, "w") as f:
    f.write(md)
print(f"MD artifact written: {MD_OUT}")

MD artifact written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/reports/artifacts/01_exploration/04_cleaning/01_04_04_identity_resolution.md


## Cell 19 -- Verification

In [19]:
assert JSON_OUT.exists(), f"FAIL: JSON artifact missing: {JSON_OUT}"
assert MD_OUT.exists(), f"FAIL: MD artifact missing: {MD_OUT}"

with open(JSON_OUT) as f:
    loaded = json.load(f)

assert "sql_queries" in loaded, "FAIL: sql_queries block missing from JSON"
assert len(loaded["sql_queries"]) >= 8, (
    f"FAIL: Expected >= 8 SQL queries, got {len(loaded['sql_queries'])}"
)
assert len(loaded["decisions"]) >= 5, (
    f"FAIL: Expected >= 5 decisions, got {len(loaded['decisions'])}"
)
assert task_c_results["status"] == "PASS", (
    f"FAIL: Task C gate failed -- duplicate count {dup_count} out of range"
)
assert verdict in ("A", "B", "C"), f"FAIL: Invalid verdict {verdict}"

print("Verification PASS:")
print(f"  JSON artifact: {JSON_OUT.name} ({JSON_OUT.stat().st_size:,} bytes)")
print(f"  MD artifact:   {MD_OUT.name} ({MD_OUT.stat().st_size:,} bytes)")
print(f"  SQL queries:   {len(loaded['sql_queries'])}")
print(f"  Decisions:     {len(loaded['decisions'])}")
print(f"  Task C: {task_c_results['status']} (duplicate_rows={dup_count})")
print(f"  Task G: Verdict={verdict}")
print("01_04_04 notebook complete.")

Verification PASS:
  JSON artifact: 01_04_04_identity_resolution.json (24,134 bytes)
  MD artifact:   01_04_04_identity_resolution.md (13,746 bytes)
  SQL queries:   14
  Decisions:     5
  Task C: PASS (duplicate_rows=489)
  Task G: Verdict=A
01_04_04 notebook complete.
